In [ ]:
from google.colab import drive
drive.mount('/content/drive')



In [ ]:
!unzip -q -o "/content/drive/MyDrive/PCGITA.zip" -d "/content/"


In [ ]:
import os

print(os.listdir("/content/PCGITA"))

In [ ]:
print(os.listdir("/content/PCGITA/Vowels"))

In [ ]:
!pip install -q transformers torchaudio

import torch
import torch.nn as nn
import torchaudio
import numpy as np
import os

from transformers import Wav2Vec2Processor, WavLMModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [ ]:
# STEP 3 — Load PC-GITA dataset

import os
import pandas as pd

DATASET_PATH = "/content/PCGITA/Vowels"

data = []

# -------------------------
# Healthy speakers
# -------------------------

control_path = os.path.join(DATASET_PATH, "Control")

for vowel in ["A","E","I","O","U"]:
    vowel_folder = os.path.join(control_path, vowel)

    for file in os.listdir(vowel_folder):
        if file.endswith(".wav"):
            data.append({
                "filename": os.path.join("Control", vowel, file),
                "label": 0
            })


# -------------------------
# Parkinson speakers
# -------------------------

pd_path = os.path.join(DATASET_PATH, "Patologicas")

for vowel in ["A","E","I","O","U"]:
    vowel_folder = os.path.join(pd_path, vowel)

    for file in os.listdir(vowel_folder):
        if file.endswith(".wav"):
            data.append({
                "filename": os.path.join("Patologicas", vowel, file),
                "label": 1
            })


labels_df = pd.DataFrame(data)

print("Total samples:", len(labels_df))
print("\nClass distribution:")
print(labels_df["label"].value_counts())

labels_df.head()

In [ ]:
from transformers import WavLMModel, Wav2Vec2FeatureExtractor

# correct processor for WavLM
processor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base")

# correct model
model_wav = WavLMModel.from_pretrained("microsoft/wavlm-base")

# freeze model
model_wav.eval()
for p in model_wav.parameters():
    p.requires_grad = False

In [ ]:
def extract_features_seq(audio_path):
    waveform, sr = torchaudio.load(audio_path)

    # mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # resample
    if sr != 16000:
        waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)

    inputs = processor(
        waveform.squeeze().numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model_wav(inputs.input_values)

    return outputs.last_hidden_state.squeeze(0)  # (T,768)

In [ ]:
X_seq = []
y = []

for _, row in labels_df.iterrows():
    path = os.path.join(DATASET_PATH, row["filename"])

    try:
        feat = extract_features_seq(path).cpu().numpy().astype(np.float16)
        X_seq.append(feat)
        y.append(row["label"])
    except:
        print("Error")

y = np.array(y)

In [ ]:
np.save("X_seq.npy", np.array(X_seq, dtype=object))
np.save("y.npy", y)



In [ ]:
X_seq = np.load("X_seq.npy", allow_pickle=True)
y = np.load("y.npy")

In [ ]:
X_train_seq, X_test_seq, y_train, y_test = train_test_split(
    X_seq, y, test_size=0.2, stratify=y
)

In [ ]:
from torch.nn.utils.rnn import pad_sequence

X_train = pad_sequence(
    [torch.tensor(x, dtype=torch.float32) for x in X_train_seq],
    batch_first=True
)

X_test = pad_sequence(
    [torch.tensor(x, dtype=torch.float32) for x in X_test_seq],
    batch_first=True
)

# 🔥 truncate
MAX_LEN = 400
X_train = X_train[:, :MAX_LEN, :]
X_test  = X_test[:, :MAX_LEN, :]

# 🔥 normalize
X_train = (X_train - X_train.mean()) / (X_train.std() + 1e-5)
X_test  = (X_test - X_test.mean()) / (X_test.std() + 1e-5)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=16,
    shuffle=True
)

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)   # (B,T,1)
        context = torch.sum(weights * x, dim=1)        # (B,H)
        return context


class GRU_Attention(nn.Module):
    def __init__(self):
        super().__init__()

        self.gru = nn.GRU(
            768,
            256,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.attn = Attention(512)

        self.fc = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        out, _ = self.gru(x)        # (B,T,512)

        context = self.attn(out)    # (B,512)

        logits = self.fc(context)
        prob = torch.sigmoid(logits)

        return context, prob

In [ ]:
model = GRU_Attention()

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)



In [ ]:
for epoch in range(30):
    model.train()
    total_loss = 0

    for Xb, yb in train_loader:
        feat, out = model(Xb)

        out = out.squeeze()
        loss = criterion(out, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss}")

In [ ]:
model.eval()

with torch.no_grad():
    _, preds = model(X_test)

preds = (preds.squeeze() > 0.5).float()

acc = (preds == y_test).float().mean()

print("🔥 Deep Model Accuracy:", acc.item())

In [ ]:
with torch.no_grad():
    train_feat, _ = model(X_train)
    test_feat, _ = model(X_test)

train_feat = train_feat.numpy()
test_feat = test_feat.numpy()

scaler = StandardScaler()
train_feat = scaler.fit_transform(train_feat)
test_feat = scaler.transform(test_feat)

svm = SVC(kernel='rbf', C=20, gamma=0.01)

svm.fit(train_feat, y_train.numpy())

y_pred = svm.predict(test_feat)

print("🔥 FINAL Accuracy (Attention + SVM):",
      accuracy_score(y_test.numpy(), y_pred))